# A9：Agent Granular Tracing — 從 Span 到根因診斷

> **對應 Blog：** FDE 面試準備指南（十一）Agent 線上除錯 + （十九）Multi-Agent 統計評估與細粒度追蹤  
> **核心目標：** 親手建 OpenTelemetry 風格的 Span Tracing，親眼看到瀑布圖，找出是哪個 Agent 拖累了整體。

## 面試情境

> 「這是一個由 4 個 Agent 組成、包含 10 次 Tool-calling 的複雜工作流。客戶說最終答案正確率很低。你如何建立統計評估管線？如何進行 Granular Tracing 抓出是哪個 Agent 或哪次 Tool-calling 出問題？」

## 學習路徑

| Part | 內容 | 關鍵觀察 |
|------|------|----------|
| 1 | Span 基礎設計 | trace_id / span_id / parent_id |
| 2 | Agent 自動注入追蹤 | 每個節點都有 Span |
| 3 | 瀑布圖視覺化 | 一眼看出瓶頸 |
| 4 | 五大故障模式診斷 | 從日誌找根因 |
| 5 | Multi-Agent 評估指標 | 四層評估框架 |
| 6 | 迴圈偵測 | action history 分析 |
| 7 | 面試 DARK 框架 | |

In [ ]:
import time
import uuid
import json
import asyncio
import random
from typing import Any, Optional
from dataclasses import dataclass, field
from collections import defaultdict

print("環境就緒")

## Part 1：Span 基礎設計（OpenTelemetry 風格）

In [ ]:
@dataclass
class Span:
    """
    OpenTelemetry 風格的 Span
    每個 Agent 節點和每次 Tool 呼叫都是一個 Span
    """
    span_id: str
    trace_id: str
    parent_id: Optional[str]       # None = root span
    name: str                       # agent/tool 的名稱
    span_type: str                  # "agent" | "tool" | "llm"
    start_time: float
    end_time: Optional[float] = None
    
    # LLM 特有指標
    input_tokens: int = 0
    output_tokens: int = 0
    model_id: str = ""
    
    # 結果
    status: str = "running"         # running | ok | error
    error_msg: str = ""
    input_summary: str = ""
    output_summary: str = ""
    
    @property
    def duration_ms(self) -> float:
        if self.end_time is None:
            return (time.time() - self.start_time) * 1000
        return (self.end_time - self.start_time) * 1000
    
    @property
    def cost_usd(self) -> float:
        """粗估成本（gpt-4o-mini 定價）"""
        return (self.input_tokens * 0.00015 + self.output_tokens * 0.0006) / 1000


class TraceCollector:
    """收集所有 Span，提供查詢和分析"""
    
    def __init__(self):
        self.spans: list[Span] = []
        self._span_map: dict[str, Span] = {}
    
    def start_span(self, name: str, span_type: str, trace_id: str,
                   parent_id: str = None, **kwargs) -> Span:
        span = Span(
            span_id=str(uuid.uuid4())[:8],
            trace_id=trace_id,
            parent_id=parent_id,
            name=name,
            span_type=span_type,
            start_time=time.time(),
            **kwargs
        )
        self.spans.append(span)
        self._span_map[span.span_id] = span
        return span
    
    def end_span(self, span: Span, status: str = "ok", error: str = "",
                 output_summary: str = "", **kwargs):
        span.end_time = time.time()
        span.status = status
        span.error_msg = error
        span.output_summary = output_summary
        for k, v in kwargs.items():
            setattr(span, k, v)
    
    def get_trace(self, trace_id: str) -> list[Span]:
        return [s for s in self.spans if s.trace_id == trace_id]
    
    def get_children(self, parent_id: str) -> list[Span]:
        return [s for s in self.spans if s.parent_id == parent_id]


collector = TraceCollector()
print("✅ Span Tracing 基礎設計完成")
print("  Span 結構：span_id, trace_id, parent_id, duration_ms, input/output_tokens, cost")

## Part 2：Multi-Agent 系統（帶自動追蹤）

In [ ]:
# 模擬 Multi-Agent 系統（Router → Legal + Finance → Synthesis）

async def router_agent(trace_id: str, parent_id: str, query: str) -> dict:
    span = collector.start_span(
        name="router_agent", span_type="agent",
        trace_id=trace_id, parent_id=parent_id,
        input_summary=query[:50], model_id="gpt-4o-mini",
        input_tokens=200
    )
    
    await asyncio.sleep(0.45)  # 模擬 LLM 決策時間
    
    result = {"route_to": ["legal", "finance"]}
    collector.end_span(span, output_summary=str(result), output_tokens=50)
    return {"result": result, "span_id": span.span_id}


async def legal_agent(trace_id: str, parent_id: str, query: str) -> dict:
    agent_span = collector.start_span(
        name="legal_agent", span_type="agent",
        trace_id=trace_id, parent_id=parent_id,
        input_summary=query[:50], model_id="gpt-4o-mini",
        input_tokens=1500
    )
    
    # Tool 1: search_contracts（模擬較慢的工具）
    tool_span = collector.start_span(
        name="search_contracts", span_type="tool",
        trace_id=trace_id, parent_id=agent_span.span_id,
        input_summary="contract keywords"
    )
    await asyncio.sleep(0.80)  # ⚠️ 慢！
    collector.end_span(tool_span, output_summary="3 contracts found")
    
    # LLM 生成
    await asyncio.sleep(0.40)
    
    result = {"legal_risk": "MEDIUM", "risk_reasons": ["缺少違約金上限"]}
    collector.end_span(agent_span, output_summary=str(result), output_tokens=400)
    return {"result": result, "span_id": agent_span.span_id}


async def finance_agent(trace_id: str, parent_id: str, query: str,
                         inject_slowness: bool = False) -> dict:
    agent_span = collector.start_span(
        name="finance_agent", span_type="agent",
        trace_id=trace_id, parent_id=parent_id,
        input_summary=query[:50], model_id="gpt-4o-mini",
        input_tokens=1200
    )
    
    # Tool 1: get_exchange_rate
    t1 = collector.start_span("get_exchange_rate", "tool", trace_id, agent_span.span_id)
    await asyncio.sleep(0.15)
    collector.end_span(t1, output_summary="USD/TWD=32.5")
    
    # Tool 2: query_erp（模擬有時極慢）
    t2 = collector.start_span("query_erp", "tool", trace_id, agent_span.span_id,
                               input_summary="Q4 revenue")
    erp_delay = 2.90 if inject_slowness else 0.30  # 注入慢查詢
    await asyncio.sleep(erp_delay)
    if inject_slowness:
        collector.end_span(t2, output_summary="{revenue: 4200000}", status="ok")
    else:
        collector.end_span(t2, output_summary="{revenue: 4200000}")
    
    # LLM 生成
    await asyncio.sleep(0.75)
    
    result = {"financial_impact": 4_200_000, "currency": "TWD", "payment_risk": "LOW"}
    collector.end_span(agent_span, output_summary=str(result), output_tokens=350)
    return {"result": result, "span_id": agent_span.span_id}


async def synthesis_agent(trace_id: str, parent_id: str,
                           legal_result: dict, finance_result: dict) -> dict:
    span = collector.start_span(
        name="synthesis_agent", span_type="agent",
        trace_id=trace_id, parent_id=parent_id,
        input_summary="legal+finance results", model_id="gpt-4o-mini",
        input_tokens=800
    )
    await asyncio.sleep(0.60)
    result = {"decision": "APPROVE_WITH_CONDITIONS", "summary": "合約可簽，但需修改第 3.2 條"}
    collector.end_span(span, output_summary=result["decision"], output_tokens=300)
    return {"result": result, "span_id": span.span_id}


print("Multi-Agent 函數定義完成")

## Part 3：執行並生成瀑布圖

In [ ]:
def render_waterfall(trace_id: str, collector: TraceCollector):
    """ASCII 瀑布圖：一眼看出哪個 Agent/Tool 最慢"""
    spans = collector.get_trace(trace_id)
    if not spans:
        print("No spans found")
        return
    
    total_start = min(s.start_time for s in spans)
    total_end = max(s.end_time for s in spans if s.end_time)
    total_duration = (total_end - total_start) * 1000  # ms
    
    BAR_WIDTH = 40
    
    print(f"\n{'Span Name':<28} {'Duration':>8} {'Tokens':>8} {'Cost':>8}   Timeline")
    print("-" * 90)
    
    # 按 start_time 排序
    for span in sorted(spans, key=lambda s: s.start_time):
        if span.end_time is None:
            continue
        
        indent = "  " if span.parent_id else ""
        if span.span_type == "tool":
            indent = "    "
        
        name = f"{indent}{span.name}"
        duration = span.duration_ms
        tokens = span.input_tokens + span.output_tokens
        cost = span.cost_usd
        
        # 時間條
        offset_pct = (span.start_time - total_start) / (total_end - total_start)
        width_pct = duration / total_duration
        bar_start = int(offset_pct * BAR_WIDTH)
        bar_width = max(1, int(width_pct * BAR_WIDTH))
        
        bar_char = "█" if span.span_type == "agent" else "░"
        if span.status == "error":
            bar_char = "✕"
        
        # 標記慢的 span
        slow_marker = " ⚠️" if duration > 1000 else ""
        
        bar = " " * bar_start + bar_char * bar_width
        print(f"{name:<28} {duration:>7.0f}ms {tokens:>7} {cost:>7.4f}$  |{bar}{slow_marker}")
    
    print("-" * 90)
    total_tokens = sum(s.input_tokens + s.output_tokens for s in spans)
    total_cost = sum(s.cost_usd for s in spans)
    print(f"{'TOTAL E2E':<28} {total_duration:>7.0f}ms {total_tokens:>7} {total_cost:>7.4f}$")


# 執行正常場景
trace_id_normal = str(uuid.uuid4())[:8]
root_span = collector.start_span("request", "root", trace_id_normal,
                                  input_summary="合約審閱請求")

print("=" * 55)
print("場景 1：正常執行（無異常）")
print("=" * 55)

router_r = await router_agent(trace_id_normal, root_span.span_id, "審閱採購合約")
legal_r, finance_r = await asyncio.gather(
    legal_agent(trace_id_normal, root_span.span_id, "法律風險分析"),
    finance_agent(trace_id_normal, root_span.span_id, "財務影響計算", inject_slowness=False)
)
synthesis_r = await synthesis_agent(trace_id_normal, root_span.span_id,
                                      legal_r["result"], finance_r["result"])

collector.end_span(root_span, output_summary="完成")
render_waterfall(trace_id_normal, collector)

In [ ]:
# 執行有問題的場景（Finance Agent 的 ERP 查詢極慢）

collector2 = TraceCollector()

# 重新定義 agents 使用 collector2
async def run_full_trace(inject_slowness: bool, trace_label: str):
    """執行完整的 Multi-Agent trace"""
    trace_id = str(uuid.uuid4())[:8]
    root = collector2.start_span("request", "root", trace_id, input_summary="合約審閱")
    
    # Router
    r_span = collector2.start_span("router_agent", "agent", trace_id, root.span_id,
                                    input_tokens=200)
    await asyncio.sleep(0.45)
    collector2.end_span(r_span, output_summary="route:[legal,finance]", output_tokens=50)
    
    # Legal + Finance 並行
    async def run_legal():
        a = collector2.start_span("legal_agent", "agent", trace_id, root.span_id, input_tokens=1500)
        t = collector2.start_span("search_contracts", "tool", trace_id, a.span_id)
        await asyncio.sleep(0.80)
        collector2.end_span(t, output_summary="3 found")
        await asyncio.sleep(0.40)
        collector2.end_span(a, output_summary="MEDIUM risk", output_tokens=400)
    
    async def run_finance():
        a = collector2.start_span("finance_agent", "agent", trace_id, root.span_id, input_tokens=1200)
        t1 = collector2.start_span("get_exchange_rate", "tool", trace_id, a.span_id)
        await asyncio.sleep(0.15)
        collector2.end_span(t1, output_summary="32.5")
        
        t2 = collector2.start_span("query_erp", "tool", trace_id, a.span_id)
        delay = 2.90 if inject_slowness else 0.30
        await asyncio.sleep(delay)
        collector2.end_span(t2, output_summary="{revenue:4200000}")
        
        await asyncio.sleep(0.75)
        collector2.end_span(a, output_summary="4.2M TWD", output_tokens=350)
    
    await asyncio.gather(run_legal(), run_finance())
    
    # Synthesis
    s = collector2.start_span("synthesis_agent", "agent", trace_id, root.span_id, input_tokens=800)
    await asyncio.sleep(0.60)
    collector2.end_span(s, output_summary="APPROVE_WITH_CONDITIONS", output_tokens=300)
    
    collector2.end_span(root, output_summary="完成")
    return trace_id


print("=" * 55)
print("場景 2：Finance Agent ERP 查詢極慢（inject slowness）")
print("=" * 55)

slow_trace_id = await run_full_trace(inject_slowness=True, trace_label="slow")

def render_waterfall_v2(trace_id: str, coll: TraceCollector):
    spans = coll.get_trace(trace_id)
    if not spans: return
    total_start = min(s.start_time for s in spans)
    total_end = max(s.end_time for s in spans if s.end_time)
    total_duration = (total_end - total_start) * 1000
    BAR_WIDTH = 40
    print(f"\n{'Span Name':<28} {'Duration':>8}   Timeline")
    print("-" * 72)
    for span in sorted(spans, key=lambda s: s.start_time):
        if not span.end_time: continue
        indent = "" if span.parent_id is None else ("  " if span.span_type == "agent" else "    ")
        name = f"{indent}{span.name}"
        duration = span.duration_ms
        offset_pct = (span.start_time - total_start) / (total_end - total_start)
        width_pct = min(1.0, duration / total_duration)
        bar_start = int(offset_pct * BAR_WIDTH)
        bar_width = max(1, int(width_pct * BAR_WIDTH))
        bar_char = "█" if span.span_type == "agent" else "░"
        slow = " ⚠️⚠️ SLOW" if duration > 1000 else ""
        bar = " " * bar_start + bar_char * bar_width
        print(f"{name:<28} {duration:>7.0f}ms   |{bar}{slow}")
    print("-" * 72)
    print(f"{'TOTAL E2E':<28} {total_duration:>7.0f}ms")

render_waterfall_v2(slow_trace_id, collector2)

print("\n診斷結論：")
print("  瓶頸在 query_erp（~2900ms）")
print("  Finance Agent 幾乎決定了整個 E2E 延遲")
print("  修復方向：加 Cache / 優化 ERP 查詢 / 升級連接")

## Part 4：五大故障模式的診斷（DARK 框架）

In [ ]:
# 模擬不同故障場景的 log，練習診斷

FAULT_LOGS = {
    "無限迴圈": """
Step 1: Action: query_database → Error: timeout
Step 2: Action: query_database → Error: timeout
Step 3: Action: query_database → Error: timeout
Step 4: Action: query_database → Error: timeout
Step 5: Action: query_database → Error: timeout
... (重複 20 次)
Step 22: [stopped: max_steps reached]
""",
    "工具描述不清": """
Step 1: Action: search({"q": "合約違約金"}) → Error: invalid_schema
Step 2: Action: search({"query": "合約違約金"}) → Error: invalid_param_name
Step 3: Action: search({"keyword": "合約"}) → Error: missing_required_field
Step 4: Thought: "工具呼叫失敗，嘗試直接回答"
Step 5: Output: "合約違約金為 500 萬（幻覺！）"
""",
    "任務偏移": """
[Turn 1] User: "幫我分析 Q4 銷售數據"
[Turn 5] Agent 開始討論 Q3 數據
[Turn 10] Agent 分析了市場趨勢（原始任務未完成）
[Turn 15] User: "你還沒分析 Q4 啊"
[Turn 15] Agent: "您好！請問您需要什麼協助？" （完全重置）
""",
    "幻覺：Retrieval 失敗": """
Step 1: Tool: search_kb("違約金條款") → results: [] (score: 0.45 < threshold: 0.7)
Step 2: Thought: "知識庫找不到，但我知道這個問題的答案"
Step 3: Output: "根據標準合約，違約金通常為合約金額的 10%" （知識庫外的資訊！）
Faithfulness Score: 0.12 (threshold: 0.8) ← 幻覺確認
"""
}

def diagnose_with_dark(fault_name: str, log: str):
    """用 DARK 框架分析故障日誌"""
    print(f"\n{'='*55}")
    print(f"故障：{fault_name}")
    print(f"{'='*55}")
    print("日誌：")
    print(log)

diagnose_with_dark("無限迴圈", FAULT_LOGS["無限迴圈"])
print("""
D（診斷）: Tool 層故障 + Agent 層缺乏錯誤處理。
  工具持續失敗，但 Agent 只知道重試，不知道換策略。

A（還需要什麼）:
  - 資料庫的 slow query log
  - 這段時間的資料庫 CPU/Connection 監控
  - 這個 query 有沒有 index

R（根本原因）: 最可能是資料庫高負載，
  加上 Agent 的 error handling 缺乏退避和降級邏輯。

K（修法）:
  ① 工具層：加 retry with exponential backoff（max 3 次）
  ② 工具層：3 次失敗後回傳結構化錯誤 {status: unavailable}
  ③ Agent 層：收到 tool_unavailable 後降級（查快取或告知用戶）
  ④ 監控：工具連續 3 次失敗 → 立即告警
""")

diagnose_with_dark("工具描述不清", FAULT_LOGS["工具描述不清"])
print("""
D: Tool description 不清楚 → LLM 猜測參數名稱 → 幻覺填補。

R: Tool Schema 沒有明確說明參數名稱和格式。

K: 更新 Tool description，明確指定參數名稱和範例：
  name: "search_kb"
  parameters:
    keyword (string): 搜尋關鍵字，必填，範例: '違約金條款'
    top_k (int, default=5): 回傳數量
""")

## Part 5：Multi-Agent 四層評估指標

In [ ]:
# 四層評估指標實作

def calculate_routing_accuracy(predictions: list[set], ground_truths: list[set]) -> dict:
    """指標 1：Routing Accuracy — Router 的分派對不對？"""
    precisions, recalls, f1s = [], [], []
    for pred, truth in zip(predictions, ground_truths):
        if not truth:
            continue
        tp = len(pred & truth)
        precision = tp / len(pred) if pred else 0
        recall = tp / len(truth) if truth else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    return {
        "precision": sum(precisions) / len(precisions) if precisions else 0,
        "recall": sum(recalls) / len(recalls) if recalls else 0,
        "f1": sum(f1s) / len(f1s) if f1s else 0
    }


def calculate_trajectory_efficiency(actual_steps: int, golden_steps: int) -> float:
    """指標 3：Trajectory Efficiency — 執行路徑有多接近最優？"""
    return golden_steps / actual_steps if actual_steps > 0 else 0


def calculate_cost_efficiency(spans: list[Span]) -> dict:
    """指標 4：Cost/Token Efficiency"""
    total_tokens = sum(s.input_tokens + s.output_tokens for s in spans)
    total_cost = sum(s.cost_usd for s in spans)
    by_agent = defaultdict(lambda: {"tokens": 0, "cost": 0.0, "duration_ms": 0})
    for s in spans:
        if s.span_type in ["agent", "tool"]:
            by_agent[s.name]["tokens"] += s.input_tokens + s.output_tokens
            by_agent[s.name]["cost"] += s.cost_usd
            by_agent[s.name]["duration_ms"] = s.duration_ms
    return {"total_tokens": total_tokens, "total_cost": total_cost, "by_agent": dict(by_agent)}


# 測試
print("=" * 55)
print("Multi-Agent 四層評估指標")
print("=" * 55)

# 1. Routing Accuracy
routing_preds = [
    {"legal", "finance"},       # 正確
    {"finance"},                # 漏了 legal
    {"legal", "finance", "hr"}, # 多了 hr
]
routing_truth = [
    {"legal", "finance"},
    {"legal", "finance"},
    {"legal", "finance"},
]
routing_metrics = calculate_routing_accuracy(routing_preds, routing_truth)
print(f"\n1. Routing Accuracy:")
print(f"   Precision: {routing_metrics['precision']:.2f}")
print(f"   Recall:    {routing_metrics['recall']:.2f} ← 0.83 代表偶爾漏掉子 Agent")
print(f"   F1:        {routing_metrics['f1']:.2f}")

# 2. Tool Success Rate
tool_calls = [
    {"tool": "search_contracts", "success": True},
    {"tool": "query_erp", "success": True},
    {"tool": "get_exchange_rate", "success": True},
    {"tool": "query_erp", "success": False},  # 失敗
    {"tool": "query_erp", "success": True},
]
success_rate = sum(1 for t in tool_calls if t["success"]) / len(tool_calls)
print(f"\n2. Tool Success Rate: {success_rate:.0%}")
erp_calls = [t for t in tool_calls if t["tool"] == "query_erp"]
erp_success = sum(1 for t in erp_calls if t["success"]) / len(erp_calls)
print(f"   query_erp success rate: {erp_success:.0%} ⚠️（低於整體）")

# 3. Trajectory Efficiency
cases = [(4, 4), (4, 5), (4, 7)]
print(f"\n3. Trajectory Efficiency:")
for golden, actual in cases:
    eff = calculate_trajectory_efficiency(actual, golden)
    print(f"   Golden={golden} steps, Actual={actual} steps → efficiency={eff:.2f}")

# 4. Cost Efficiency
trace_spans = collector.get_trace(trace_id_normal)
cost_metrics = calculate_cost_efficiency(trace_spans)
print(f"\n4. Cost Efficiency:")
print(f"   Total Tokens: {cost_metrics['total_tokens']:,}")
print(f"   Total Cost: ${cost_metrics['total_cost']:.4f}")
print(f"   Per-Agent breakdown:")
for agent_name, metrics in cost_metrics['by_agent'].items():
    print(f"   └── {agent_name}: {metrics['tokens']} tokens, ${metrics['cost']:.4f}, {metrics['duration_ms']:.0f}ms")

## Part 6：迴圈偵測（Action History 分析）

In [ ]:
from collections import deque

class LoopDetector:
    """基於 Action History 的迴圈偵測"""
    
    def __init__(self, window: int = 5):
        self.window = window
        self.action_history: deque[str] = deque(maxlen=window)
        self.step_count = 0
    
    def add_action(self, action: str) -> dict:
        self.step_count += 1
        self.action_history.append(action)
        
        history = list(self.action_history)
        
        # 偵測 1：全部一樣（相同動作重複）
        if len(history) >= 3 and len(set(history)) == 1:
            return {"is_loop": True, "type": "repeated_action",
                    "action": "INJECT_HINT" if self.step_count <= 15 else "FORCE_STOP"}
        
        # 偵測 2：交替重複（A B A B A）
        if len(history) >= 4:
            half = len(history) // 2
            if history[:half] == history[half:half*2]:
                return {"is_loop": True, "type": "alternating_loop",
                        "action": "FORCE_STOP"}
        
        # 硬性步驟上限
        if self.step_count > 20:
            return {"is_loop": True, "type": "max_steps_exceeded",
                    "action": "FORCE_STOP"}
        
        return {"is_loop": False, "action": "CONTINUE"}


# 測試不同的 action 序列
print("=" * 55)
print("迴圈偵測測試")
print("=" * 55)

scenarios = [
    ("正常進展", ["tool_A", "tool_B", "tool_C", "llm_think", "tool_D"]),
    ("相同動作重複", ["query_db", "query_db", "query_db", "query_db", "query_db"]),
    ("交替迴圈", ["tool_A", "tool_B", "tool_A", "tool_B", "tool_A"]),
]

for scenario_name, actions in scenarios:
    detector = LoopDetector(window=5)
    print(f"\n場景：{scenario_name}")
    for i, action in enumerate(actions, 1):
        result = detector.add_action(action)
        status = f"⚠️ {result['type']} → {result['action']}" if result["is_loop"] else "✅ 正常"
        print(f"  Step {i}: {action:20s} → {status}")
        if result["is_loop"]:
            break

print("\n護欄設計建議：")
print("""
step <= 10  → 繼續執行
step 11~15  → 注入提示：「請評估你是否已有足夠資訊可以回答」
step > 15   → 強制輸出當前最佳答案
step > 20   → 硬性終止，回傳 fallback 訊息
""")

## Part 7：面試 DARK 框架 + 評估指標速查

In [ ]:
print("""
面試答題框架：DARK
─────────────────────────────────────────────────────
D → Diagnose   根據症狀，最可能是哪種故障模式？
A → Ask        你還需要什麼資訊才能確認？（要主動問）
R → Root Cause 推斷最可能的根本原因
K → Kill it    怎麼修，以及怎麼預防再次發生

追蹤架構（面試問「你怎麼設計可觀測性」）：
  用 OpenTelemetry 為 Graph 中的每個 Node 和 Tool Call 注入 Span
  每個 Span 記錄：
  ├── span_id, trace_id, parent_id（建立父子關係）
  ├── duration_ms（延遲）
  ├── input/output_tokens（成本）
  ├── status（ok / error）
  └── input/output_summary（可用於 Debug）
  統一收集到 Cloud Trace / LangSmith / Phoenix

Multi-Agent 四層評估：
  1. Routing Accuracy       Router 分派 Precision / Recall / F1
  2. Tool Execution Rate    工具呼叫成功率（分工具統計）
  3. Trajectory Efficiency  |Golden Steps| / |Actual Steps|
  4. Cost/Token per Task    追蹤趨勢（上升 = Context 累積問題）

診斷流程：
  Metrics → 哪個 Agent 分數低
  Trace → Drill down 那個 Agent 的 Span
  Root Cause → 哪類 Query + 哪個 Tool + 什麼具體錯誤
  Fix + Regression Test → 加進 CI/CD，部署前自動跑
─────────────────────────────────────────────────────
""")